# 강의 06 · 실습 11 — 패턴 5 오케스트레이터-워커 · (2-2) 빈칸 채우기 II

## 1. 문제상황

- 인사팀은 사내 안내문을 주제별로 씁니다. 안내문은 주제마다 문단 수가 다릅니다.
- 두 문단이면 되는 주제도 있고 네 문단이 필요한 주제도 있어서, 담당자는 틀을 미리 정하지 못합니다.
- 담당자는 주제를 받으면 먼저 어떤 문단이 필요한지 정하고, 문단마다 따로 쓴 뒤, 순서대로 이어 붙입니다.
- 문단을 정하는 일과 문단마다 쓰는 일이 주제마다 반복되고, 문단 수가 달라질 때마다 작업 방식도 달라집니다.

## 2. 문제와 목표

- **문제**: 하위 작업(문단)의 개수와 내용이 입력(주제)에 따라 달라지므로, 병렬 경로 수를 코드에 미리 그려 둘 수 없습니다.
- **목표**: 주제를 입력하면 orchestrator 노드가 필요한 문단 계획을 구조화 출력으로 세우고, 계획된 문단 수만큼 worker 노드를 실행 시점에 팬아웃해 문단마다 쓰게 한 뒤, synthesizer 노드가 전부 모아 안내문으로 이어 붙이는 처리 흐름을 만듭니다.
    - orchestrator 노드: 주제를 보고 문단 계획(제목 목록)을 구조화 출력으로 세웁니다.
    - worker 노드: 문단 하나를 쓰는 노드이며, 계획한 문단 수만큼 `Send`로 팬아웃됩니다.
    - synthesizer 노드: 모델을 부르지 않고 문단 부분 결과를 이어 붙여 안내문을 만듭니다.
- **목표 달성 여부의 판정 기준**: 주제를 입력했을 때, 계획한 문단 수만큼 문단이 따로 만들어지고, 그 수만큼의 부분 결과가 합쳐져 안내문이 되는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex11_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 계획 규격을 정의합니다.**
    - 계획 규격 `Section`은 문단 제목(`name`)과 내용 한 줄(`description`)을 가지고, `Sections`는 `Section`의 목록을 가집니다.
    - 부모 상태 `GuideState`는 주제(`topic`), 문단 계획(`sections`), 쓴 문단(`written`), 완성 안내문(`guide`) 키 네 개를 가지며, `written`에는 리스트를 이어 붙이는 `operator.add` 리듀서를 겁니다.
    - worker 전용 상태 `WorkerState`는 자기 문단(`section`)과 같은 리듀서를 건 `written` 키 두 개만 가집니다.
2. **orchestrator 노드를 만듭니다.**
    - `Sections` 규격을 건 모델을 불러 「사내 안내문 문단 계획을 필요한 만큼(2~4개) 세운다」는 지침으로 주제의 문단 계획을 받고, 그 목록을 상태의 `sections` 키에 씁니다.
3. **worker 노드를 만듭니다.**
    - worker는 `WorkerState`를 받아 자기 문단의 제목과 내용 한 줄을 모델에 넣고 「받은 문단 하나를 두 문장으로 쓴다」는 지침으로 문단을 쓴 뒤, 제목 줄을 붙인 부분 결과 하나를 리스트에 담아 `written` 키로 돌려줍니다.
4. **synthesizer 노드를 만듭니다.**
    - synthesizer는 모델을 부르지 않고, 상태의 `written` 부분 결과 전부를 빈 줄로 이어 붙여 `guide` 키에 씁니다.
5. **그래프에 노드를 등록합니다.**
    - orchestrator·worker·synthesizer 세 노드를 이름과 함께 등록합니다.
    - worker는 한 번만 등록합니다.
6. **엣지를 연결합니다.**
    - START에서 orchestrator로 가는 고정 엣지를 추가합니다.
    - orchestrator 뒤에는 판단 함수 assign_workers가 `sections`의 항목마다 `Send("worker", {"section": 항목})`을 만들어 돌려주는 조건부 엣지를 추가합니다.
    - worker 뒤에는 synthesizer를, synthesizer 뒤에는 END를 고정 엣지로 연결합니다.
7. **그래프를 컴파일하고 실행합니다.**
    - 주제와 빈 계획, 빈 부분 결과 목록, 빈 안내문을 넣어 실행한 뒤, 팬아웃된 worker의 수와 완성 안내문을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 부모 상태·worker 전용 상태·계획 규격을 선언하고 리듀서를 겁니다 | `class GuideState(TypedDict)`, `Annotated[list, operator.add]` | 1 |
| ② 노드 함수 정의 | 계획을 세우는 orchestrator, 부분 결과 하나를 쓰는 worker, 합치는 synthesizer를 만듭니다 | `llm.with_structured_output(Sections)`, `def worker(state: WorkerState) -> dict` | 2, 3, 4 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 세 노드를 등록합니다 | `StateGraph(GuideState)`, `add_node` | 5 |
| ④ 엣지 연결 | 실행 시점에 worker를 팬아웃하는 Send 목록과 고정 순서를 정합니다 | `add_conditional_edges`, `Send("worker", {...})`, `add_edge` | 6 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 주제를 넣어 실행합니다 | `compile()`, `invoke()` | 7 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
# 여기에 단계 0(라이브러리 불러오기, .env 읽기, 모델 준비)을 작성합니다.

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 노드들이 읽고 쓰는 키를 선언합니다. 여러 worker가 같은 `written` 키에 동시에 쓰므로 리듀서가 반드시 필요합니다. `Annotated[list, operator.add]`가 값을 덮어쓰지 않고 이어 붙입니다. 리듀서가 없으면 한 단계에 한 값만 받는다는 오류로 실행이 멈춥니다. `WorkerState`는 worker 하나가 받는 작업 단위만 담은 별도 상태입니다. worker는 전체 상태가 아니라 자기 문단 하나와 결과 키만 봅니다.

In [ ]:
# 여기에 단계 ①(계획 규격 Section·Sections, 부모 상태 GuideState, worker 상태 WorkerState 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4)

- orchestrator는 `with_structured_output(Sections)`로 계획을 파이썬 객체로 받습니다. 출력이 자유 문장이 아니라 문단 목록이라는 규격이며, 목록의 길이가 곧 worker의 수가 됩니다.
- worker는 자기 작업 단위 하나만 처리하고 결과를 리스트 항목 하나로 돌려줍니다. 돌려준 값은 리듀서가 이어 붙입니다.
- synthesizer는 모델을 부르지 않습니다. 상태에 모인 부분 결과를 문자열로 이어 붙이기만 합니다.

In [ ]:
# 여기에 단계 ②(orchestrator, worker, synthesizer 노드 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 5)

`StateGraph`에 부모 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. worker 노드는 한 번만 등록합니다. 몇 개로 팬아웃될지는 등록이 아니라 실행 시점의 `Send` 목록이 정합니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 6)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`의 판단 함수가 노드 이름 대신 `Send` 목록을 돌려주면, 목록의 항목 수만큼 그 노드가 팬아웃됩니다. `Send` 한 쌍은 워커 이름과 그 워커가 받을 상태입니다. 세 번째 인자는 이 분기가 닿을 수 있는 노드를 그래프에 알려 주는 목록입니다. worker 뒤에는 synthesizer를, synthesizer 뒤에는 END를 고정 엣지로 연결합니다.

In [ ]:
# 여기에 단계 ④(Send 목록을 돌려주는 판단 함수와 엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 7)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 주제와 빈 값들을 넣으면 최종 상태가 돌아옵니다. 실행 중 orchestrator와 worker가 출력하는 진입 줄로 계획 수와 팬아웃된 worker 수를 대조합니다. 아래에서는 사내 메신저 보안 수칙 안내를 주제로 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. `[orchestrator] 진입` 줄에 출력된 계획 문단 수와 `[worker] 진입` 줄의 수가 같습니다.
2. `[synthesizer] 진입` 줄이 worker 줄들 뒤에 한 번만 출력되고, 합친 부분 결과 수가 worker 수와 같습니다.
3. 최종 상태의 `written`에 worker 결과가 전부 담겨 있고, 안내문은 문단 제목 줄과 두 문장이 계획 수만큼 이어져 있습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. 실행이 `InvalidUpdateError`로 멈추면 단계 ①의 `written` 리듀서를, worker가 하나만 돌면 단계 ④의 판단 함수가 `Send` 목록을 돌려주는지 다시 봅니다.